# RegimeShift-ML research companion

## tl;dr

The summary below is computed from the selected saved run. Synthetic runs validate the software only. Change `REGIMESHIFT_RUN` to inspect another completed study.

### Load saved artifacts

Use the project's Python 3.11 environment. No training occurs here.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
from IPython.display import Markdown, display

from regimeshift.reporting import interpretation, plot_results
from regimeshift.utils import read_json

get_ipython().run_line_magic("matplotlib", "inline")
pl.Config.set_tbl_rows(24)
pl.Config.set_float_precision(4)

default_run = (Path.cwd().parent if Path.cwd().name == "report" else
               Path.cwd().parent / "runs/demo" if Path.cwd().name == "notebooks" else
               Path("runs/demo"))
RUN = Path(os.environ.get("REGIMESHIFT_RUN", str(default_run))).resolve()
summary = read_json(RUN / "evaluation/summary.json")
display(Markdown(f"**Source:** {summary['source_kind']} | **Units:** {summary['target_units']}"))
display(Markdown(interpretation(summary)))


## Context & Methods

The question is whether regimes improve adaptive blending of frozen Ridge and LightGBM experts. Compare the chosen static reference and a global adaptive control. Models use 60% of dates; tuning, calibration and final evaluation use 10%, 10%, 20%.

### Key Assumptions

Labels arrive on the next observed date. Today's regime uses yesterday's completed summary. Hyperparameters stay frozen during final evaluation. Bootstrap intervals resample contiguous dates on the realized prediction path; they do not rerun model fitting. Nominal prediction-interval coverage is not guaranteed under dependence or shifts.

## Data

The dataset manifest records the file fingerprint, missingness, and date counts. Real Optiver rows stay local. Synthetic labels are simulated response functions, not actual 60-second returns.

Source terms: [Optiver competition rules](https://www.kaggle.com/competitions/optiver-trading-at-the-close/rules).

In [ ]:
manifest = read_json(RUN / "dataset.json")
splits = read_json(RUN / "splits.json")
display(pl.DataFrame([{ "block": k, "dates": len(v), "first": v[0], "last": v[-1]} for k, v in splits.items()]))
display(pl.DataFrame([{"rows": manifest["audit"]["rows"], "valid_targets": manifest["audit"]["valid_targets"], "stocks": manifest["audit"]["stocks"]}]))

## Results

All model comparisons use the same finite-target rows. The static reference was selected on tuning dates, before final evaluation.

In [ ]:
display(pl.read_parquet(RUN / "evaluation/metrics.parquet").sort("mae"))
figures = plot_results(RUN)
plt.close("all")
display(figures["model_comparison"])

### Performance over time

Daily error trajectories show whether the aggregate comparison hides unstable performance.

In [ ]:
display(figures["daily_errors"])

### Adaptation and regime support

Weights change only after a prior date's labels are revealed. Cluster IDs are descriptive, not economic ground truth. The global control removes regime conditioning while retaining online weighting.

In [ ]:
display(figures["weights_regimes"])
display(pl.read_parquet(RUN / "evaluation/regime_metrics.parquet").select("regime_id", "model", "count", "scored_dates", "mae"))

### Calibration

Interpret coverage together with interval width. A wider interval can improve coverage without improving point forecasts.

In [ ]:
display(figures["intervals"])
display(pl.read_parquet(RUN / "evaluation/interval_metrics.parquet").select("model", "interval", "count", "coverage", "mean_width"))

### Paired uncertainty

Differences are regime-aware minus reference MAE. Intervals use the configured contiguous-date moving blocks and preserve row-count weighting.

In [ ]:
display(figures["paired_differences"])

## Takeaways

This is a study of selecting among frozen experts. It cannot establish gains from retraining, intraday detection, trading profits, or a novel learning algorithm. Real-data research remains pending when this notebook uses synthetic data.

In [ ]:
display(Markdown(interpretation(summary)))